# Job offers: build

Second half of the job-offer build: joins the mappings to the preprocessed offers and exports the
dataset. Run it after the five mapping notebooks, which read `mapping/preprocessed_jobs.json` and
write one file each, keyed on `id_j`:

| notebook | output |
|---|---|
| `1_translate_job_title.ipynb` | `mapping/translated.csv` |
| `2_translate_skills.ipynb` | `mapping/translated_skills.csv` |
| `3_extract_job_title.ipynb` | `mapping/occupation_map.csv` (+ `_detailed`) |
| `4_extract_skill_mapping.ipynb` | `mapping/{KB}_skill_map.csv` (+ `_detailed`), one per KB in `KBS` |
| `5_extract_languages.ipynb` | `mapping/language_map.csv` (+ `_detailed`) |

`KBS` lists the KBs joined to the offers, one skill column each (`esco_skills`, `onet_skills`);
each must have been mapped by `4_extract_skill_mapping.ipynb`.

The job title (`3_extract_job_title.ipynb`) and the job-category aggregation use ESCO only; only
the skills depend on the KB. Languages are dropped from every KB column; `languages` holds only
the ones `5_extract_languages.ipynb` found in the description.

`id_j` comes from the deduplication in `0_preprocess_jobs.ipynb`, so these files describe that
population only.

Exports `job_offer_database.json` and the 12-category subset
`datasets/findhr/source/filtered_jobs_dt.json`.

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import sys; sys.path.append("../../../")
import pandas as pd

from ast import literal_eval
from pathlib import Path
from typing import List

from kb_graph import KBGraph

DATA_DIR = Path("../../../datasets/findhr")

KBS = ["ESCO", "ONET"]

In [ ]:
# One graph per KB: the language split of each skill column runs on its own graph.
# ESCO also generalizes the occupations.
GRAPH_CACHE = {"ESCO": "occSkill_graph.pkl", "ONET": "onet_occSkill_graph.pkl"}
graphs = {kb: KBGraph.load(Path(f"../../../cache/{GRAPH_CACHE[kb]}")) for kb in KBS}
esco_g = graphs["ESCO"]   # the job-category aggregation stays ESCO-only

In [ ]:
def filter_lang(list_skills: List[str], g: KBGraph = esco_g):
    """Split languages out of a skill list: returns (non-language skills, generalized languages)."""
    langs = {i for i in list_skills if g.is_language(label=i)}
    return sorted(set(list_skills) - langs), sorted(g.generalize_lang(label=i) for i in langs)

def generalize_occ(occ: str):
    return esco_g.ancestor_at_level(esco_g.get_uri(occ), level=2, labeled=True)

In [ ]:
jobs_dt = pd.read_json("mapping/stage1_jobs.json")
print(f"{len(jobs_dt)} offers from mapping/stage1_jobs.json")
jobs_dt.head(10)

In [ ]:
lang_map = pd.read_csv(
    "mapping/language_map.csv",
    converters={"languages": literal_eval}
)
print("len", len(lang_map))
lang_map.head(10)

In [ ]:
# One mapping file per KB, keyed on id_j; its `esco_skills` column is renamed to the KB's column.
SKILL_COLS = {kb: f"{kb.lower()}_skills" for kb in KBS}
skill_maps = {}
for kb, col in SKILL_COLS.items():
    path = Path(f"mapping/{kb}_skill_map.csv")
    assert path.exists(), f'{path} is missing -- run 4_extract_skill_mapping.ipynb with KB = "{kb}"'
    skill_maps[kb] = (pd.read_csv(path, converters={"esco_skills": literal_eval, "skills_list": literal_eval})
                      .rename(columns={"skills_list": "raw_skills", "esco_skills": col}))
    print(f"{kb}: {len(skill_maps[kb])} rows -> {col}")

skill_maps[KBS[0]].head(10)

In [ ]:
# `raw_skills` is the same input for every KB: take it once and check the others against it
# (a mismatch means a mapping was computed on an old translation).
raw = {kb: m.set_index("id_j")["raw_skills"].apply(frozenset).sort_index() for kb, m in skill_maps.items()}
for kb in KBS[1:]:
    assert raw[kb].equals(raw[KBS[0]]), \
        f'{kb}_skill_map.csv was mapped on a different skill list than {KBS[0]}_skill_map.csv ' \
        f'-- delete it and re-run 4_extract_skill_mapping.ipynb with KB = "{kb}"'

jobs_dt = jobs_dt.merge(skill_maps[KBS[0]][["id_j", "raw_skills"]], on="id_j", how="left")
assert jobs_dt["raw_skills"].notna().all(), "skill map misses offers -- re-run 4_extract_skill_mapping.ipynb"

for kb, col in SKILL_COLS.items():
    jobs_dt = jobs_dt.merge(skill_maps[kb][["id_j", col]], on="id_j", how="left")
    jobs_dt[col] = jobs_dt[col].apply(lambda x: x if isinstance(x, list) else [])

# Drop the languages from every KB column, or skill fitness would count them: `languages`
# keeps only the ones 5_extract_languages.ipynb found in the description.
for kb, col in SKILL_COLS.items():
    jobs_dt[col] = jobs_dt[col].apply(lambda x: filter_lang(x, graphs[kb])[0])

jobs_dt = jobs_dt.merge(lang_map[["id_j", "languages"]], on="id_j", how="left")
assert jobs_dt["languages"].notna().all(), "language_map.csv misses offers -- re-run 5_extract_languages.ipynb"

FROZEN = ["raw_skills", "languages", *SKILL_COLS.values()]
jobs_dt[FROZEN] = jobs_dt[FROZEN].map(frozenset)

In [ ]:
jobs_dt

In [ ]:
# No KB column may keep a language.
for kb, col in SKILL_COLS.items():
    assert all(not graphs[kb].is_language(label=i) for i in jobs_dt[col].explode().drop_duplicates().dropna())

In [ ]:
occupation_map = pd.read_csv("./mapping/occupation_map.csv")
print("len", len(occupation_map))
occupation_map.head(10)

In [ ]:
jobs_dt = jobs_dt.merge(occupation_map[["id_j", "raw_occupation", "esco_occupation"]], on="id_j", how="left")
assert jobs_dt["esco_occupation"].notna().all(), "occupation_map has empty rows -- re-run 3_extract_job_title.ipynb"
jobs_dt["job_category"] = jobs_dt["esco_occupation"].apply(generalize_occ)

In [ ]:
manual_fix = {
    "Information and communications technology professionals": "ICT",
    "Business and administration associate professionals": "Business and administration",
    "Business and administration professionals": "Business and administration",
    "Administrative and commercial managers": "Business and administration",
    "Sales workers": "Sales and customer service",
    "Science and engineering professionals": "Science and engineering",
    "Information and communications technicians": "ICT",
    "Production and specialised services managers": "Management (production and services)",
    "Metal, machinery and related trades workers": "Skilled trades",
    "Labourers in mining, construction, manufacturing and transport": "Elementary and manual labour",
    "Personal service workers": "Personal and protective services",
    "Electrical and electronic trades workers": "Skilled trades",
    "Science and engineering associate professionals": "Science and engineering",
    "Hospitality, retail and other services managers": "Management (production and services)",
    "Health professionals": "Health and care",
    "Customer services clerks": "Sales and customer service",
    "Drivers and mobile plant operators": "Transport and machine operation",
    "Stationary plant and machine operators": "Transport and machine operation",
    "Cleaners and helpers": "Elementary and manual labour",
    "Health associate professionals": "Health and care",
    "Legal, social and cultural professionals": "Legal, social and cultural",
    "General and keyboard clerks": "Clerical support",
    "Food processing, wood working, garment and other craft and related trades workers": "Skilled trades",
    "Teaching professionals": "Education",
    "Personal care workers": "Health and care",
    "Building and related trades workers, excluding electricians": "Skilled trades",
    "Protective services workers": "Personal and protective services",
    "Legal, social, cultural and related associate professionals": "Legal, social and cultural",
    "Numerical and material recording clerks": "Clerical support",
    "Food preparation assistants": "Elementary and manual labour",
    "Other clerical support workers": "Clerical support",
    "Agricultural, forestry and fishery labourers": "Elementary and manual labour",
    "Chief executives, senior officials and legislators": "Business and administration",
    "Assemblers": "Transport and machine operation",
    "Street and related sales and service workers": "Sales and customer service",
    "Refuse workers and other elementary workers": "Elementary and manual labour",
    "Handicraft and printing workers": "Skilled trades",
    "Market-oriented skilled forestry, fishery and hunting workers": "Agriculture, forestry and fishery",
    "Non-commissioned armed forces officers": "Armed forces",
    "Market-oriented skilled agricultural workers": "Agriculture, forestry and fishery",
}
jobs_dt["job_category_agg"] = jobs_dt["job_category"].apply(lambda x : manual_fix.get(x,x))
jobs_dt.head(10)

In [ ]:
translated_map = pd.read_csv("mapping/translated.csv")
print("len", len(translated_map))
translated_map.head(10)

In [ ]:
jobs_dt = jobs_dt.merge(translated_map[["id_j","eng_description"]], on="id_j", how="left")
assert jobs_dt["eng_description"].notna().all(), "translated.csv has empty rows -- re-run 1_translate_job_title.ipynb"
jobs_dt.head(10)

In [ ]:
# Rename normalized columns and export the final feature set used downstream.
jobs_dt = (
    jobs_dt
    .rename(columns={"min_experience": "years_experience", "studies_level_min": "edu_eqf"})
    [["id_j", "job_category_agg", "job_category", "raw_occupation", "esco_occupation", "years_experience", "edu_eqf", "languages", "raw_skills", *SKILL_COLS.values(), "eng_description"]]
)

In [ ]:
assert not jobs_dt.duplicated().any()
assert not jobs_dt.isna().any().any()

In [ ]:
jobs_dt.head(10)

In [ ]:

for col in ["languages", "raw_skills", *SKILL_COLS.values()]:
    jobs_dt[col] = jobs_dt[col].apply(sorted)

jobs_dt.to_json("job_offer_database.json", orient="records", indent=2)

## Subset of the job offers

In [ ]:
subset = [
    'Business and administration',
    'Clerical support',
    'Science and engineering','ICT',
    'Sales and customer service',
    'Legal, social and cultural',
    'Elementary and manual labour',
    'Personal and protective services',
    'Health and care',
    'Management (production and services)',
    'Skilled trades',
    'Education'
]

jobs_dt = pd.read_json("job_offer_database.json")
filtered_jobs_dt = jobs_dt[jobs_dt.job_category_agg.isin(subset)]

# Written to the experiment's data folder, where 1_application.ipynb and the rest read it.
(DATA_DIR / "source").mkdir(parents=True, exist_ok=True)
filtered_jobs_dt.to_json(DATA_DIR / "source/filtered_jobs_dt.json", orient="records", indent=2)
print(f"{len(filtered_jobs_dt)} / {len(jobs_dt)} offers -> {DATA_DIR    / 'source/filtered_jobs_dt.json'}")

In [ ]:
filtered_jobs_dt.job_category_agg.value_counts().to_dict()